In [1]:
import ee
import geemap
import os
import geopandas as gpd
import time, math

from datetime import datetime, timedelta
import pandas as pd
import re
import calendar

In [2]:
ee.Authenticate()
ee.Initialize()

In [ ]:
import logging
logger = logging.getLogger()
logger.setLevel(logging.ERROR)  # 设置记录级别为 ERROR 或更高级别
file_handler = logging.FileHandler('warnings.log')
logger.addHandler(file_handler)
    
def fmask(image):
    cloudsBitMask = (1 << 3)
    cloudshadowBitMask = (1 << 4)
    snowBitMask = (1 << 5)

    qaMask = image.select('BQA').bitwiseAnd(cloudsBitMask).eq(0) \
                        .And(image.select('BQA').bitwiseAnd(cloudshadowBitMask).eq(0)) \
                        .And(image.select('BQA').bitwiseAnd(snowBitMask).eq(0))

    saturationMask = image.select('QA_RADSAT').eq(0)

    opticalBands = image.select(['Blue', 'Green', 'Red', 'Nir', 'Swir1', 'Swir2']).multiply(0.0000275).add(-0.2)

    return image.addBands(opticalBands, None, True).updateMask(qaMask).updateMask(saturationMask)

def cal_cloud(image):
    cloudsBitMask = (1 << 3)
    cloudshadowBitMask = (1 << 4)
    snowBitMask = (1 << 5)

    qaMask = image.select('BQA').bitwiseAnd(cloudsBitMask).eq(0) \
                        .And(image.select('BQA').bitwiseAnd(cloudshadowBitMask).eq(0)) \
                        .And(image.select('BQA').bitwiseAnd(snowBitMask).eq(0))
    cloudMask = qaMask.lt(1)

    cloudCoverage = cloudMask.reduceRegion(
                        reducer = ee.Reducer.mean(),
                        geometry = geometry,
                        scale = 120,
                        maxPixels = 1e9,
                    )
    return image.set('cloud_coverage', cloudCoverage.get('BQA'))
    
def zero_to_025_l8(image):
    condition = image.select("Nir").gt(0) \
    .And(image.select("Nir").lt(0.25));
    return image.updateMask(condition)

def process_image_collection(image_collection):
    def func_shf(image):
        image = ee.Image(image)
        date = ee.Date(image.get('system:time_start'))
        dateString = date.format('YYYY-MM-dd')
        return image.set('dateString', dateString)

    grouped = image_collection.toList(image_collection.size()).map(func_shf)

    distinctDates_list = []

    def func_gdv(image):
        dateString = ee.Image(image).get('dateString')
        return dateString

    distinctDates_list = grouped.map(func_gdv)

    distinctDates = distinctDates_list.distinct()

    def getMeanImageByDate(dateString):
        imagesOnDate = grouped.filter(ee.Filter.eq('dateString', dateString))
        meanImage = ee.ImageCollection(imagesOnDate).reduce(ee.Reducer.mean())
        return meanImage.set('system:time_start', ee.Date(dateString).millis())

    def func_ssk(dateString):
        meanImage = getMeanImageByDate(dateString)
        return meanImage.set('date', dateString)

    meanImagesList = distinctDates.map(func_ssk)
    meanImages = ee.ImageCollection(meanImagesList)

    return meanImages

bn8 = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'QA_PIXEL', 'QA_RADSAT']
bn7 = ['SR_B1', 'SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7', 'QA_PIXEL', 'QA_RADSAT']
bn5 = ['SR_B1', 'SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7', 'QA_PIXEL', 'QA_RADSAT']
bns = ['uBlue', 'Blue', 'Green', 'Red', 'Nir', 'Swir1', 'Swir2', 'BQA', 'QA_RADSAT']
ls7 = ee.ImageCollection("LANDSAT/LE07/C02/T1_L2").select(bn7, bns)
ls8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").select(bn8, bns)
ls5 = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2").select(bn5, bns)
merged = ls7.merge(ls8).merge(ls5)

index_l57 = {
    'ndwi': lambda image: image.normalizedDifference(['Green', 'Nir']).rename('ndwi'),
    'mndwi': lambda image: image.normalizedDifference(['Green', 'Swir1']).rename('mndwi'),
    'AWEIsh': lambda image: image.expression(
        'BLUE + 2.5 * GREEN - 1.5 * (NIR + SWIR1) - 0.25 * SWIR2', {
            'BLUE': image.select('Blue'),
            'GREEN': image.select('Green'),
            'NIR': image.select('Nir'),
            'SWIR1': image.select('Swir1'),
            'SWIR2': image.select('Swir2')
        }).float().rename('AWEIsh')}

shp_file = r"D:\YR_ArcGis\YellowRiverResplit.shp"
gdf = gpd.read_file(shp_file)

selectedBands_l57 = ['Blue_mean', 'Green_mean', 'Red_mean', 'Nir_mean', 'Swir1_mean', 'Swir2_mean']

output_folder = r"C:\Users\方慈弘\Desktop\YellowRiverSplit"
os.makedirs(output_folder, exist_ok=True)

for num in range(120,150): 
    filtered_gdf = gdf[gdf['OBJECTID_1'] == num]
    geometry = geemap.geopandas_to_ee(filtered_gdf)

    all_data = []
    for year in range(1983,2025):
        start_time = time.time()
        image_all_year = (
            merged
            .filterBounds(geometry)
            .filterDate(str(year) + '-01-01', str(year) + '-12-30')
            .map(fmask)
            .map(zero_to_025_l8)
            .map(cal_cloud)
            .filter(ee.Filter.lt('cloud_coverage', 0.5))
        )
        water_image = image_all_year.median().clip(geometry)
        AWEIsh = index_l57['AWEIsh'](water_image)
        AWEIsh_method = AWEIsh.gt(0)
        image_all_year = process_image_collection(image_all_year)
        try:
          for i in range(image_all_year.size().getInfo()):
            start_time = time.time()
            image = ee.Image(image_all_year.toList(image_all_year.size()).get(i)).clip(geometry)
            date = image.get('date').getInfo()
            date_ls = ee.Date(image.get('system:time_start').getInfo())

            image_final = image.updateMask(AWEIsh_method).clip(geometry) # 双重提水

            era5 = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR') \
                .filterBounds(geometry) \
                .filter(ee.Filter.date(date_ls.advance(-1, 'day'), date_ls.advance(1, 'day'))) \
                .first().clip(geometry).reduceRegion(reducer=ee.Reducer.mean(),geometry=geometry,scale=1000).getInfo()

            temp = era5.get('temperature_2m')-273.15
            prec = era5.get('total_precipitation_sum')
            wind_u = era5.get('u_component_of_wind_10m')
            wind_v = era5.get('v_component_of_wind_10m')
            wind = math.sqrt(wind_u**2+wind_v**2)
            solar = era5.get('surface_net_solar_radiation_sum')
            evaporation = era5.get('evaporation_from_open_water_surfaces_excluding_oceans_sum')
            lai_h = era5.get('leaf_area_index_high_vegetation')
            lai_l = era5.get('leaf_area_index_low_vegetation')

            era_year = date_ls.get('year').getInfo()
            era_month = date_ls.get('month').getInfo()
            start_date = datetime(era_year, era_month, 1)
            end_date = datetime(era_year, era_month, calendar.monthrange(era_year, era_month)[1])
            
            era5_mon = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR') \
                 .filterBounds(geometry) \
                 .filter(ee.Filter.date(start_date, end_date)) \
                 .first().clip(geometry).reduceRegion(reducer=ee.Reducer.mean(),geometry=geometry,scale=1000).getInfo()
              
            temp_mon = era5_mon.get('temperature_2m')
            prec_mon = era5_mon.get('total_precipitation_sum')
            wind_u_mon = era5_mon.get('u_component_of_wind_10m')
            wind_v_mon = era5_mon.get('v_component_of_wind_10m')
            wind_mon = math.sqrt(wind_u_mon**2+wind_v_mon**2)
            solar_mon = era5_mon.get('surface_net_solar_radiation_sum')
            evaporation_mon = era5_mon.get('evaporation_from_open_water_surfaces_excluding_oceans_sum')
            lai_h_mon = era5_mon.get('leaf_area_index_high_vegetation')
            lai_l_mon = era5_mon.get('leaf_area_index_low_vegetation')

            result_median = image_final.select(selectedBands_l57).reduceRegion(
                reducer = ee.Reducer.median(),
                geometry = geometry,
                scale = 120,
                maxPixels = 1e13,
            ).getInfo()
            median = [round(value, 4) for value in result_median.values()]
    
            iteration_time = time.time() - start_time

            result = [round(temp,2), round(prec*1000,2), round(wind,2), int(solar), round(evaporation,4), round(lai_h,2), round(lai_l,2)]

            result_mon = [round(temp_mon,2), round(prec_mon*1000,2), round(wind_mon,2), int(solar_mon), round(evaporation_mon,4),
                          round(lai_h_mon,2), round(lai_l_mon,2)]

            print(num, year, date, *median, *result, *result_mon, round(iteration_time,2))

            row = [num, year, date, *median, *result, *result_mon, round(iteration_time, 2)]
            all_data.append(row)
        except:
            continue
    
    columns = [
        "num", "year", "date", "Blue", "Green", "Red", "Nir", "Swir1", "Swir2",
        "temp", "prec", "wind", "solar", "evaporation", "lai_h", "lai_l",
        "temp_mon", "prec_mon", "wind_mon", "solar_mon", "evaporation_mon", "lai_h_mon", "lai_l_mon", "iteration_time"]
    
    df = pd.DataFrame(all_data, columns=columns)
    output_file = os.path.join(output_folder, f"Reach_{num}.xlsx")
    df.to_excel(output_file, index=False)

120 1986 1986-04-26 0.1248 0.1805 0.2056 0.2074 0.1928 0.1587 9.92 0.22 2.66 11696974 -0.0001 0 0.51 283.31 0.66 1.23 511094211 -0.0038 0 0.51 11.88
120 1986 1986-06-13 0.1471 0.1991 0.2478 0.232 0.1651 0.1375 19.24 0.86 1.06 18088304 -0.0006 0 0.51 296.68 8.6 2.11 568402326 -0.009 0 0.51 9.87
120 1986 1986-06-29 0.1246 0.177 0.2374 0.2123 0.1428 0.1317 22.69 0 2.76 22205626 -0.0003 0 0.52 296.68 8.6 2.11 568402326 -0.009 0 0.51 11.12
120 1986 1986-07-15 0.1048 0.1665 0.228 0.2133 0.0367 0.0282 27.03 0 4.68 16205239 -0.0001 0 0.52 298.15 24.53 0.85 588273422 -0.0236 0 0.52 9.4
120 1986 1986-07-31 0.0862 0.149 0.1675 0.1919 0.0145 0.0103 21.65 0.11 1.82 19604293 -0.0025 0 0.52 298.15 24.53 0.85 588273422 -0.0236 0 0.52 10.4
120 1986 1986-11-04 0.0913 0.1396 0.1373 0.1611 0.1143 0.0909 1.93 0.0 2.95 9960150 -0.0002 0 0.51 271.19 1.2 1.04 244975938 -0.0028 0 0.51 9.89
120 1986 1986-06-20 0.0917 0.1498 0.1743 0.1885 0.0418 0.0359 22.16 0.01 1.73 20168376 -0.0001 0 0.51 296.68 8.6 2.11 5684